# CreditMetrics

### Pytania teoretyczne:

#### Wyjaśnić pojęcie ratingów, jakie poziomy ratingów stosują najbardziej znane agencje ratingowe?

Ad: Ratingi to system oceny wiarygodności kredytowej instytucji / państwa. Jest nadawany przez agencję ratingową na podstawie
oceny ryzyka ekonomicznego, politycznego i społecznego.
Firmy /Pańswta którym są wystawiane noty można podzielić głownie na 2 grupy:
- standard inwestycyjny (bezpieczniejsze, ale mniejsze zyski)
- standard spekulacyjny (ryzkowniejsze, ale większe zyski)

Takie agencje ratingowe powinny być obiektywnym (nie mogą firmy / pańswa na nie naciskać) i stabilnym (ratingi nie powinny zmieniać się zbyt często, bo niesie to ze sobą konswekwencję) podmiotem oceniającym.

System skąłda się z not (na podstawie S&P i Fitch, Moody’s zapisuje noty w zbliżony sposób):
- AAA (najlepsza nota)
- AA
- A
- BBB
- BB
- B
- CCC
- D (bankructwo - niewypłacalność)

Dodatkowo są oceny pośrednie czyli tak jak w skzole -5/+5 tak tutaj mamy np. A-, BB+ itp.

Te agencję także na podsatwie swoich historyczych danych tworzą także swoje tablice przejść, czyli tablice ktore nas informują jakie jest prawdopodobieństwo, że firma zmieni swój rating w czasie X na rating Y, oraz druga tabela, która mówi nam jakie jest prawdopodobieństwo bankróctwa w przeciągu następnych X lat.


#### Wyjaśnić ideę modelu CreditMetrics. Na czym on bazuje i w jaki sposób opisuje ryzyko kredytowe?

Ad: Model CreditMetrics został opracowany w celu określania ryzyka kredytowego nie tylko pojedynczych ekspozycji, ale całego ich portfela. Model ten opiera się na całościowej koncepcji ryzyka kredytowego – w przeciwieństwie do uproszczonych modeli, wycenia on ryzyko uwzględniając nie tylko samo prawdopodobieństwo bankructwa, ale też ewentualne pogorszenie (lub poprawę) ratingu kredytowego w wybranym horyzoncie czasu.
CreditMetrics bazuje na:Prawdopodobieństwach migracji do różnych klas ratingowych (tzw. macierz przejścia), bazujących na danych historycznych. Zgodnie z wyciągiem w Twoim pliku .csv, dla obligacji o ratingu np. BBB szansa pozostania w klasie wynosi aż 88,48%, podczas gdy szansa bankructwa wynosi zaledwie 0,15%.Przyszłych krzywych zwrotu (stóp procentowych) dla poszczególnych klas ratingowych, używanych do wyceny (dyskontowania) instrumentów na wypadek zmiany ich ratingu.Stopach odzysku kapitału (recovery rates), które określają, jaką część środków można odzyskać w przypadku faktycznego ogłoszenia niewypłacalności podmiotu (Default).

### Zadanie praktyczne:
- Z wykorzystaniem modelu CreditMetrics wyznaczyć 99,9% VaR i ES dla portfela składającego się z portfela trzech poniżej opisanych obligacji skarbowych:
	- 3-letnia obligacja o ratingu BBB (cena wykupu 100 000 zł, subordinated),
	- 5-letnia obligacja o ratingu B (cena wykupu 50 000 zł, senior secured),
	- 2-letnia obligacja o ratingu CCC (cena wykupu 50 000 zł, senior unsecured).
- Opisać kolejne kroki rozwiązywanego zadania.
- Zbadać, w jaki sposób zmieniłyby się wyniki, jeśli założylibyśmy, że inwestycje w portfelu są skorelowane (tj. współczynniki korelacji między inwestycjami w portfelu wynoszą: r12 = 0.2, r13 = 0.15, r23 = 0.4). Jak uzyskane wyniki łączą się z pojęciem "dywersyfikacji ryzyka".
- Na podstawie dokumentacji technicznej CreditMetrics na przykładzie opisać, jakie są sposoby wyznaczania korelacji w modelu (podobno roździał nr 9 z ang).

# Opis rozwiązania (punkt 2 polecenia)

Przyjmujemy horyzont 1 rok. Wartość portfela po roku zależy od tego, jakie ratingi będą mieć emitenci naszych obligacji. Rating może się zmienić zgodnie z **macierzą migracji**, a nowy rating oznacza inną krzywą stóp dyskontowych (czyli inną cenę obligacji). Jeśli rating spadnie do *Default* - bankructwo, dostajemy tylko część nominału zgodnie ze stopą odzysku (zależną od seniority).

### Kroki rozwiązania
1. **Wczytanie macierzy migracji** z orginalnego pliku z `dane_cm.xlsx`.
2. **Zapisanie ze zdjęc danych** (stopy zwrotu forward, stopy odzysku) — bierzemy wartości zapisane z excelu (na zdjęciach). Są to standardowe stopy z dokumentacji CreditMetrics.
3. **Zdefiniowanie portfela** — 3 obligacje (rating, nominał, lata do wykupu, seniority).
4. **Monte Carlo (N symulacji)** — dla każdej iteracji losujemy nowe ratingi dla 3 obligacji i wyceniamy je po roku, następnie sumujemy do wartości portfela.
5. **Metryki ryzyka** — Obliczamy E[V], kwantyl 0.1%, VaR(99.9%), ES(99.9%).
6. **Wykresy** — Dodatkowe wykresy - histogram portfela, rozkłady per obligacja, empiryczna CDF (by sprawdzić czy pokrywa się wraz z oficialnymi danymi).

#### Założenie
Inwestycje niezależne (brak korelacji). Założenie to dodatkowo omawiam w punkcie 4 praktycznego zadania, czyli metody wyznaczania korelacji w CreditMetrics.

In [13]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Wczytuję macierz migracji z pliku xlsx
raw = pd.read_excel('dane_cm.xlsx', sheet_name='Arkusz1', header=None)
ratings = ['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC']
# Def to Default, czyli bankructwo
destinations = ratings + ['Def']
migration = pd.DataFrame(
    # 1-8 to od AAA do CCC (Wiersze od 1 do 7)
    # 1-9 to od AAA do Def (kolumny od 1 do 8)
    # Indeks 0 na wierszach to indeks pythonowy, dlatego bierzemy od indeksu 1 a nie 0, a znów dla kolumn 0 to pusty nagłówek
    raw.iloc[1:8, 1:9].values.astype(float),
    index=ratings,
    columns=destinations
)

print("Macierz migracji (prawdopodobieństwa przejścia do nowego ratingu po 1 roku):")
display(migration)

Macierz migracji (prawdopodobieństwa przejścia do nowego ratingu po 1 roku):


,AAA,AA,A,BBB,BB,B,CCC,Def
AAA,0.9340,0.0594,0.0064,0.0000,0.0002,0.0000,0.0000,0.0000
AA,0.0161,0.9055,0.0746,0.0026,0.0009,0.0001,0.0000,0.0002
A,0.0007,0.0228,0.9244,0.0463,0.0045,0.0012,0.0001,0.0000
BBB,0.0005,0.0026,0.0551,0.8848,0.0476,0.0071,0.0008,0.0015
BB,0.0002,0.0005,0.0042,0.0516,0.8691,0.0591,0.0024,0.0129
B,0.0000,0.0004,0.0013,0.0054,0.0635,0.8422,0.0191,0.0681
CCC,0.0000,0.0000,0.0000,0.0062,0.0205,0.0408,0.6920,0.2405


### stopy zwrotu; stopa odzysku, a klasa obligacji

Przepisuje tu na sztywno stopy zastosowane w pliku dostarczonych danych. Dodatkowo sprawdziłem czy faktycznie takie same są podane w stndardach **CreditMetrics Technical Document**, z którego korzystalśmy także na zajęciach.
Tabel z **CreditMetrics Technical Document** (Gupton, Finger, Bhatia, 1997, J.P. Morgan / RiskMetrics Group):

- **One-year forward zero curves by credit rating category** (Tabela 2.3 dokumentacji technicznej). Te same wartości są używane w przykładach w rozdziale 3 *("Stand-alone value distributions")* oraz w rozdziale 9 *("Analytic portfolio calculation")*, ostatni z nich to rozdział, z którego korzystamy w naszym zadaniu.
- **Recovery rates by seniority class** (Tabela 7.1 dokumentacji technicznej) — oryginalnie z badań Carty, Lieberman / Moody's (1996): *"Corporate Bond Defaults and Default Rates"*.

In [14]:
# Stopy zwrotu [%] — lata 1..4 dla każdej klasy ratingowej
yields = pd.DataFrame([
    [3.60, 4.17, 4.73, 5.12],
    [3.65, 4.22, 4.78, 5.17],
    [3.72, 4.32, 4.93, 5.32],
    [4.10, 4.67, 5.25, 5.63],
    [5.55, 6.02, 6.78, 7.27],
    [6.05, 7.02, 8.03, 8.52],
    [15.05, 15.02, 14.03, 13.52],
], index=ratings, columns=[1, 2, 3, 4]) / 100

# Stopy odzysku wg seniority (% nominału w razie default)
recovery = {
    'Senior Secured':      0.5380,
    'Senior Unsecured':    0.5113,
    'Senior Subordinated': 0.3852,
    'Subordinated':        0.3274,
    'Junior Subordinated': 0.1709,
}

print("Stopy zwrotu (forward):")
print((yields * 100).round(2))
print("\nStopy odzysku:")
for k, v in recovery.items():
    print(f"  {k}: {v:.2%}")

Stopy zwrotu (forward):
         1      2      3      4
AAA   3.60   4.17   4.73   5.12
AA    3.65   4.22   4.78   5.17
A     3.72   4.32   4.93   5.32
BBB   4.10   4.67   5.25   5.63
BB    5.55   6.02   6.78   7.27
B     6.05   7.02   8.03   8.52
CCC  15.05  15.02  14.03  13.52

Stopy odzysku:
  Senior Secured: 53.80%
  Senior Unsecured: 51.13%
  Senior Subordinated: 38.52%
  Subordinated: 32.74%
  Junior Subordinated: 17.09%


In [15]:
# Definiujemy portfel z 3 obligacji
# years_left = lata do wykupu po horyzoncie 1 rok (czyli obecna długość - 1)
portfolio = [
    {'name': 'BBB 100k Subordinated',       'rating': 'BBB', 'capital': 100_000, 'years_left': 2, 'seniority': 'Subordinated'},
    {'name': 'B 50k Senior Secured',        'rating': 'B',   'capital': 50_000,  'years_left': 4, 'seniority': 'Senior Secured'},
    {'name': 'CCC 50k Senior Unsecured',    'rating': 'CCC', 'capital': 50_000,  'years_left': 1, 'seniority': 'Senior Unsecured'},
]

# Wypisujemy konfigurację każdej z obligacji
for b in portfolio:
    print(f"  {b['name']}: rating={b['rating']}, nominał={b['capital']:,} zł, "
          f"lata do wykupu ={b['years_left']}, seniority={b['seniority']}")

  BBB 100k Subordinated: rating=BBB, nominał=100,000 zł, lata do wykupu =2, seniority=Subordinated
  B 50k Senior Secured: rating=B, nominał=50,000 zł, lata do wykupu =4, seniority=Senior Secured
  CCC 50k Senior Unsecured: rating=CCC, nominał=50,000 zł, lata do wykupu =1, seniority=Senior Unsecured


In [ ]:
# Symulacja Monte Carlo (bez korelacji) + metryki ryzyka
np.random.seed(42)
# Ilość symulacji, ustawiamy tak jak na przykładzie z wykładów -> 10_000
N = 10_000
K = len(portfolio)

# Pusta tablica na wyniki wypełniona zerami na start. W nich będzie trzymać sumę wartości portfela dla kaz1dej symulacji
bond_values = np.zeros((N, K))
# Pusta tablica na dodatkowe informacje jaki rating został wylooswany w danej symulacji dla danej obligacji
ratings_drawn = np.empty((N, K), dtype=object)

# Iterujemy po każdej obligacji (mamy ich 3) i losujemy po N wariantów nowego ratingu dla danej obligacji. Nie używamy standardowej pętli, po to by przyspieszyć proces oraz sam uczę się optymalizacji kodu w kierunku bardziej wektorowego tam gdzie tylko się da
for k, bond in enumerate(portfolio):
    # # Wyciągamy z macierzy przejść odpowiedni wiersz dla obecnej obligacji i jej ratingu
    # # Dostajemy wiersz z prawdopodobieństwami przejścia w inny rating dla tego ratingu 
    probs = migration.loc[bond['rating']].values
    
    # Tworzę progi na rozkładzie normalnym N(0,1)
    cum_probs = np.cumsum(probs)

    # Progi wyznaczają granice: od -inf do Z1 to 'AAA', od Z1 do Z2 to 'AA' itd.
    # Ostatni próg (dla 'Def') to zazwyczaj bardzo niski wynik Z.
    thresholds = stats.norm.ppf(cum_probs[:-1]) # Nie bierżemy ostatniego, bo to 1.0 -> +inf
    
    # Losujemy z rozkładu normalnego
    z_scores = np.random.normal(0, 1, N)
    
    # Mapujemy wylosowane wartości Z na konkretne ratingi
    # Szukamy, w którym przedziale progów wpadła wartość z_score, by odpowiedni odobrać nowe ratingi dla obligacji
    new_ratings_idx = np.digitize(z_scores, thresholds)
    new_ratings = np.array([destinations[i] for i in new_ratings_idx])
    
    # Zapisujemy nasz wylosowany wiersz nowych ratingów dla tej obligacji do naszej pomocniczje aggregującej zmiennej.
    # Dzięki temu mamy już zbiór N wariantów dla danej obligacji mówiący jaką będzie miała rating za rok
    ratings_drawn[:, k] = new_ratings

    # Teraz będziemy wyciągać wycenę obligacji na następny rok
    # Najpierw wyciągamy informację ile lat zostało do wykupu naszej obligacji
    years_left = bond['years_left']
    
    # Wyciągmy informację jaka stopa zwrotu jest dla tej obligacji z jej ratingiem (przy jej sprzedaniu za rok, dlatego polega to na years_left, bo wtedy patrzymy na kolejny rok a nie obecny)
    yield_map = yields[years_left].to_dict()
    
    # - Default → nominał × stopa odzysku
    # - inny rating → nominał / (1 + forward_yield)^lata_do_wykupu
    # Teraz zmieniamy nasze wylosowane nazwy ratingów na prawidłowe zwroty
    # Czyli zamieniamy nazwę przyszłego raitngu np AAA na jego stopę zwortu dla danego roku
    y_vec = np.array([yield_map.get(r, np.nan) for r in new_ratings])
    
    # Na sam koniec zliczamy faktyczną wartość stopy zwrotu, bazując na tym czy emitent naszej obligcji zbankrutował, czy nie
    # Jeśli nie zbankrutował to nowa wartośći wyliczana jest za pomocą wzoru na dyskontowanie, czyli V = N / (1+r)^t, gdzie N - wartość nominalna, r to stopa dla naszego ratingu, t to liczba lat do końca 
    # Natomiast jeśli zbankrutował to nowa wartośći to V = N * RR, gdzie N tak jak wcześniej jako nominalna wartość natomiast RR to stopa odzysku
    bond_values[:, k] = np.where(
        new_ratings == 'Def',
        bond['capital'] * recovery[bond['seniority']],
        bond['capital'] / (1 + y_vec) ** years_left
    )

# Liczymy wartość portfela, czyli suma 3 obligacji w każdej iteracji
portfolio_values = bond_values.sum(axis=1)

# Teraz liczymy nasze metryki ryzyka
# Prosta wartość oczekiwana
EV = portfolio_values.mean()
# Kwantyl rzędu 0.001, czyli 0.01%
# Szukamy wartości portfela, poniżej której znalazło się tylko 0.1% najgorszych scenariuszy (czyli 10 najgorszych przypadków na 10 000).
Q = np.quantile(portfolio_values, 0.001)
# Proste liczenie VaR na poziomie ufności 99.9%.
# VaR = Wartość oczekiwana - Kwantyl.
# Mówi jak dużo możemy stracić względem tego, co spodziewaliśmy się zarobić (średniej - EV), w najgorszym scenariuszu przez nas wybranym, czyli tym kwantylu 0.01%
VaR_999 = EV - Q
# Tak jak na pooprzednim przedmiocie, mówi "Jeśli już przekroczymy poziom VaR, to ile średnio wtedy stracimy"
ES_999 = EV - portfolio_values[portfolio_values < Q].mean()

print(f"Wartość oczekiwana portfela  E[V]:   {EV:>12,.2f} zł")
print(f"Kwantyl 0.1%                  Q:     {Q:>12,.2f} zł")
print(f"VaR (99.9%) = E[V] - Q:              {VaR_999:>12,.2f} zł")
print(f"ES  (99.9%) = E[V] - mean(V<Q):      {ES_999:>12,.2f} zł")

Wartość oczekiwana portfela  E[V]:     165,970.15 zł
Kwantyl 0.1%                  Q:       112,251.49 zł
VaR (99.9%) = E[V] - Q:                 53,718.66 zł
ES  (99.9%) = E[V] - mean(V<Q):         69,504.06 zł


In [17]:
fig = go.Figure()

# Główny histogram
fig.add_trace(go.Histogram(
    x=portfolio_values, nbinsx=100, marker_color='#3498db', opacity=0.7, name='Rozkład portfela'
))

# Ogon strat
tail = portfolio_values[portfolio_values < Q]
fig.add_trace(go.Histogram(
    x=tail, nbinsx=30, marker_color='#e74c3c', opacity=0.8, name='Ogon strat (V < Q)'
))

# Pionowe linie
fig.add_vline(x=EV, line_color='green', line_width=2, line_dash='solid')
fig.add_vline(x=Q, line_color='red', line_width=2, line_dash='dash')
fig.add_vline(x=EV-ES_999, line_color='orange', line_width=2, line_dash='dash')

# Linie do legendy
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', line=dict(color='green', width=2, dash='solid'), name=f'E[V] = {EV:,.0f} zł'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', line=dict(color='red', width=2, dash='dash'), name=f'Q(0.1%) = {Q:,.0f} zł'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', line=dict(color='orange', width=2, dash='dash'), name=f'E[V] - ES = {EV-ES_999:,.0f} zł'))

fig.update_layout(
    title=f'Rozkład wartości portfela ({N:,} symulacji Monte Carlo)',
    xaxis_title='Wartość portfela po 1 roku [zł]',
    yaxis_title='Liczba scenariuszy (skala logarytmiczna - nie pokazywał się ogon strat bez niej)',
    yaxis_type='log',
    barmode='overlay',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.05,
        bordercolor="Black",
        borderwidth=1
    ),
    annotations=[
        dict(
            x=0.02, y=0.97, xref='paper', yref='paper',
            text=f'VaR (99.9%) = {VaR_999:,.0f} zł<br>ES  (99.9%) = {ES_999:,.0f} zł',
            showarrow=False, bgcolor='white', bordercolor='black', borderwidth=1, align='left'
        )
    ]
)
fig.show()


In [18]:
fig = go.Figure()
colors = ['#2ecc71', '#f39c12', '#e74c3c']

for k, bond in enumerate(portfolio):
    fig.add_trace(go.Histogram(
        x=bond_values[:, k], nbinsx=80, opacity=0.55, marker_color=colors[k], name=bond['name']
    ))
    
    nominal = bond.get('capital', bond.get('face')) 
    current_yield = yields.loc[bond['rating'], bond['years_left']]
    current_value = nominal / (1 + current_yield) ** bond['years_left']
    
    fig.add_vline(x=current_value, line_color=colors[k], line_width=2, line_dash='dash')
    
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines', 
        line=dict(color=colors[k], width=2, dash='dash'), 
        name=f"Start {bond['name']}: {current_value:,.0f} zł"
    ))

fig.update_layout(
    title='Rozkład wartości poszczególnych obligacji',
    xaxis_title='Wartość obligacji po 1 roku [zł]',
    yaxis_title='Liczba scenariuszy',
    yaxis_type='log',
    barmode='overlay',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.05,
        bordercolor="Black",
        borderwidth=1
    )
)
fig.show()


In [19]:
import plotly.express as px
import plotly.graph_objects as go

# Obliczam dzisiejszą (początkową) wartość całego portfela
# Sumujemy dzisiejsze CENY RYNKOWE każdej z 3 obligacji
current_portfolio_value = 0
for bond in portfolio:
    nominal = bond.get('capital', bond.get('face'))
    # Stopa dla obecnego ratingu na pełny czas (dzisiaj to years_left + 1 rok horyzontu)
    t_today = bond['years_left'] + 1
    yield_today = yields.loc[bond['rating'], t_today] if t_today in yields.columns else yields.loc[bond['rating'], bond['years_left']]
    
    current_portfolio_value += nominal / (1 + yield_today) ** t_today

# Tworzymy wykres Dystrybuanty
fig = px.ecdf(x=portfolio_values)
fig.update_traces(line_color='#9b59b6', line_width=2)

# Liczymy prawdopodobieństwo zysku (Value > Dzisiejsza Wartość)
prob_loss = np.mean(portfolio_values < current_portfolio_value)
prob_profit = 1 - prob_loss

# Dodajemy linię PROGU RENTOWNOŚCI - czyli tej obecnej wyceny
fig.add_vline(x=current_portfolio_value, line_color='red', line_dash='dash', line_width=2)

fig.add_annotation(
    x=current_portfolio_value,
    y=prob_loss,
    text=f'Próg rentowności: {current_portfolio_value:,.0f} zł<br>Szansa na zysk: {prob_profit:.1%}',
    showarrow=True,
    arrowhead=2,
    ax=100,
    ay=40,
    bgcolor="white",
    bordercolor="black"
)

fig.update_layout(
    title='Empiryczna Skumulowana Dystrybuanta wartości portfela',
    xaxis_title='Wartość portfela po 1 roku [zł]',
    yaxis_title='Skumulowane prawdopodobieństwo (P(V < x))',
    showlegend=False
)
fig.show()


### Jak interpretować wykres

Powyższy wykres skumulowany odpowiada na pytanie o nasze ryzyko i szanse portfela. Linia pokazuje nam, jak bardzo prawdopodobne jest, że wylądujemy poniżej konkretnej kwoty zaznaczonej na osi X, czyli np w okolicach 150 tys. zł widzimy że dystrybuanta pokazuje nam ~3% szans, że nasza wartość portfela będzie niższa bądź równa tym 150 tys. zł.
Co tak naprawdę mówi nam jak prawdopdobne jest że wartość naszego portfela spadnie na taką stratną wycenę.

Im krzywa dystrybuanty na wykresie jest bardziej stroma i przesunięta w prawo, tym bezpieczniejszy jest nasz portfel (oraz idealnie by kumulacja tej dystrybuanty była na prawo od naszego progu rentowności). Dzięki temu wykres jasno pokazuje nam wyliczoną procentową szansę na to, że nasz portfel po symulowanym roku nie przyniesie straty względem zainwestowanego kapitału początkowego.

In [20]:
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=[f"Bond {k+1}: {bond['rating']}" for k, bond in enumerate(portfolio)],
    shared_yaxes=True
)

for k, bond in enumerate(portfolio):
    counts = pd.Series(ratings_drawn[:, k]).value_counts().reindex(destinations, fill_value=0)
    emp = counts.values / N
    theo = migration.loc[bond['rating']].values

    # Słupki teoretyczne (brane z macierzy)
    fig.add_trace(
        go.Bar(
            x=destinations, 
            y=theo, 
            name='Teoretyczne (macierz)', 
            marker_color='#3498db', 
            opacity=0.8,
            showlegend=(k == 0)
        ),
        row=1, col=k+1
    )

    # Słupki empiryczne (wynik losowania Monte Carlo)
    fig.add_trace(
        go.Bar(
            x=destinations, 
            y=emp, 
            name='Empiryczne (MC)', 
            marker_color='#e67e22', 
            opacity=0.8,
            showlegend=(k == 0)
        ),
        row=1, col=k+1
    )

fig.update_yaxes(type="log", row=1, col=1, title_text="Prawdopodobieństwo (log)")
fig.update_yaxes(type="log", row=1, col=2)
fig.update_yaxes(type="log", row=1, col=3)

fig.update_layout(
    title_text='Porównanie: rozkład wylosowanych ratingów vs macierz migracji',
    barmode='group',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
        bordercolor="Black",
        borderwidth=1
    ),
    height=450
)

fig.show()

## Metody wyznaczania korelacji w CreditMetrics

Na podstawie **CreditMetrics Technical Document** (rozdział 8 *"Asset value model"* i rozdział 9 *"Analytic portfolio calculation"*) korelacje w modelu dotyczą nie bezpośrednio ratingów, ale wartości aktywów emitentów. 

Do oszacowania samej korelacji aktywów CreditMetrics proponuje kilka metod (od najdokładniejszej do najprostszej):

### 1. Bezpośrednia korelacja zwrotów akcji (equity returns)
Dla dużych emitentów giełdowych korelację aktywów przybliżamy korelacją historycznych zwrotów ich akcji. Proste i dokładne, ale wymaga, żeby oba podmioty były notowane na giełdzie, by mieć te dane do liczenia. Sam także używam korelacji (a dokłądniej korelacji spadków, po to by połączyć instrumenty ktore mają spadki w różnych momentach) w swoich algorytmach do budowania portfeli ze startegii i aktywów. Wychodzi to dość bardzo skutecznie i mocno redukuje obsunięcia kapitału. 

### 2. Podejście indeksowe (industry / country indices)
Gdy nie mamy szeregów zwrotów pojedynczych firm, każdego emitenta mapujemy na miks indeksów sektorowych i krajowych z wagami. Korelację dwóch emitentów liczymy z macierzy korelacji indeksów i ich wag. Czyli jest to poodbne do opcji nr. 1 z zwrotami, z tym, że nasze zwrotu, które nie istnieją zastępujemy bardziej ogólnymi z sektora z ktroego pochodzi spółka. Jednak jeszcze warto zaznaczyć, że nic tu nie jest czarno białe, dlatego używamy wag, bo często duże spółki to nie tylko pojedynczy sektor

Przykład: emitent A = 60% indeks chemiczny PL oraz 40% indeks energetyczny PL; emitent B = 100% indeks chemiczny US. Korelację A–B obliczamy jako ważoną kombinację korelacji tych indeksów.

A tutaj przykład jak dokąłdnie liczy się tą korelację ważoną:
Firma A zależy od Indeksu 1 z wagą wA. Firma B zależy od Indeksu 2 z wagą wB. Dodatkowo same indeksy są ze sobą powiązane korelacja r12.

Wtedy wzór wygląda tak: Korelacja firm A i B = wA * wB * r12

Na przkładzie liczb:
Firma A zależy od ropy w 60% (wA = 0.6).
Firma B zależy od technologii w 40% (wB = 0.4).
Korelacja ropy z technologią wynosi 50% (r12 = 0.5).
Wynik: Korelacja między firmami A i B = 0.6 * 0.4 * 0.5 = 0.12 (czyli 12%).

To pokazuje, że firmy są połączone tylko o tyle, o ile ich indeksy są ze sobą połączone.


### 3. Model czynnikowy (factor model)
Zwrot firmy dekomponujemy na:
- część systemową — zależną od kilku czynników (np. indeksów sektorowych, krajowych, globalnych). Czyli jak np idzie kryzys, to wtedy cały sektor dany spada, nie tylko pojedyncza forma z tego sektoru
- część idiosynkratyczną — specyficzną dla firmy, nieskorelowaną z innymi emitentami. To co dzieje się tylko wewnątrz firmy (np. pożar w fabryce, mądry prezes, który rozwinął firmę). To jest unikalne dla danej firmy i nie ma nic wspólnego z innymi.

Korelacja między firmami wynika wtedy tylko z ich wspólnej ekspozycji na czynniki systemowe. Im większa część idiosynkratyczna, tym niższa korelacja. Im bardziej firma zależy od samej siebie, a mniej od gospodarki, tym trudniej o jej powiązanie (korelację) z innymi firmami.

### 4. Stała korelacja (constant correlation)
Najprostsze podejście: wszystkie pary emitentów mają tę samą korelację aktywów (CreditMetrics używa np. 0.30 w przykładach rozdziału 9). Używane, gdy brakuje danych o sektorach/krajach lub portfel jest mały. Nieststy jednak w dużym portfelu prowadzi to do nierealistycznej struktury zależności (niedoszacowanie lub przeszacowanie ryzyka koncentracji).

### 5. Od korelacji aktywów do joint transition matrix
Mając korelację aktywów ρ, dla pary emitentów wyznaczamy macierz łącznych przejść (8×8 dla 8 ratingów) przez całkowanie dwuwymiarowej gęstości normalnej nad prostokątnymi obszarami wyznaczonymi przez progi.

Przykład: Mając dwie firmy i ich wzajemną korelację (np. 12% z przykładu wyżej), robimy tak:

Dla każdej firmy osobno z macierzy migracji wyznaczamy progi. Czyli to to są liczby, które mówią nam, w prosty sposób "Jeśli wynik firmy jest między 1.5 a 2.5, to dostaje rating A". Czyli typowe progi.
Te progi nakładamy na siebie na jednym wykresie. Progi firmy A idą pionowo, a progi firmy B idą poziomo.
W ten sposób powstaje wspomniana siatka 8x8, co daje nam 64 małe prostokąty.
Każdy taki prostokąt to inna para wyników (np. A bankrutuje, B zostaje w AAA).
Za pomocą funkcji całki z rozkładu normalnego, sprawdzamy, jakie jest prawdopodobieństwo, że wylosowany punkt wpadnie do konkretnego prostokąta.
Wynik: Jeśli dla prostokąta "Firma A: BBB, Firma B: BBB" wyjdzie nam z tej całki liczba 0.82, to znaczy, że szansa na to, że obie firmy po roku będą miały rating BBB, wynosi 82%.